# Advanced Python Booleans — Another 15 Fully Worked Problems

## An incremental, tutorial-style problem notebook

This is a **new companion** to the introductory Booleans lesson, not a revision of the previous 20-problem notebook. Its rhythm follows the supplied lesson: introduce one question, write a small class or expression, test it, explain *why* Python behaves that way, improve the design, and verify the final solution. Many explanatory Markdown cells deliberately separate short code experiments.

**Topics:** `bool` versus `int`; ordinary and integer enums; JSON and string input; regular-expression matches; file end-of-stream; weak references; flag masks; chained and reflected comparisons; structural pattern matching; non-finite numbers; dictionary views; unstable boolean hooks; coroutines; and recursive truth testing.

**Requirements:** Python 3.10+ and the standard library only. Run all cells **from top to bottom**. Problem-specific names are prefixed to avoid accidental cross-problem interference. Every problem ends with assertions and a small follow-up demonstration. No external files, network calls, optional libraries, or nonstandard dependencies are required.

## How to work through the notebook

1. Read the **scenario** and formulate your prediction before each probe.
2. Execute one code cell at a time, comparing actual behavior with the explanation that follows.
3. Read the **solution construction** rather than skipping straight to the final assertions.
4. Change one input at a time: zero versus nonzero, empty versus absent, alive versus expired, and so on.
5. A verification cell prints `Problem NN: passed` when its assertions succeed.

**Compact protocol reminder (from the original lesson):** Python uses `__bool__` first, falls back to `__len__` if there is no `__bool__`, and otherwise considers the object true. The `__bool__` hook must return an actual `bool`. We will now examine *consequences of these rules in different APIs*, rather than reimplement that introductory sequence.

---
## Problem 01 — A boolean is also an integer — but should your API accept it?

**Scenario.** An API accepts an integer retry count. A caller accidentally passes `True`. The value passes a naive integer check and can collide with integer dictionary keys.

**Your task.** Discover the exact type relationship and implement a validator that accepts nonnegative integers but rejects boolean values.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Start with type checks

Predict the results. `isinstance` considers inheritance; `type(value) is int` demands an exact built-in integer. Also observe that booleans participate in arithmetic.

In [1]:
print(isinstance(True, int))
print(type(True) is int)
print(True + True, False + 10)
print((True == 1), (False == 0))

True
False
2 10
True True


**Explanation.** The class `bool` is a subclass of `int`: `isinstance(True, int)` is true and `True + True` evaluates to `2`. This inheritance is often convenient in arithmetic, but it makes an `isinstance(value, int)` input contract more permissive than intended.

### Step 2 — Inspect a collision that affects real data

Dictionary keys compare equal when `True` and `1` are used; equal keys also have equal hashes. Assigning the second key replaces the value stored under the first key.

In [2]:
p01_keys = {True: 'approved'}
p01_keys[1] = 'one retry'
print(p01_keys)
print('number of keys:', len(p01_keys))
print('same hash:', hash(True) == hash(1))

{True: 'one retry'}
number of keys: 1
same hash: True


**Interpretation.** There is only one key, not two. Do not use booleans and integer identifiers interchangeably in a mapping unless this equality is explicitly intended. Converting both to strings would also lose type information in a different way; validate the inputs instead.

### Step 3 — Build the complete validator

Use `type(value) is int` when the contract explicitly means the built-in integer type, excluding `bool` and any integer subclasses. After validating the type, check the numeric range.

In [3]:
def p01_retry_count(value: object) -> int:
    if type(value) is not int:
        raise TypeError('retry count must be a plain int, not bool or another type')
    if value < 0:
        raise ValueError('retry count must be nonnegative')
    return value

print(p01_retry_count(0))
print(p01_retry_count(3))

0
3


### Step 4 — Test both valid and invalid inputs

A focused test should verify the positive contract and check specific exception classes. Do not silently turn bad input into zero.

In [4]:
assert p01_retry_count(0) == 0
assert p01_retry_count(7) == 7
for p01_invalid, p01_error in [(True, TypeError), (False, TypeError), (2.0, TypeError), (-1, ValueError)]:
    try:
        p01_retry_count(p01_invalid)
    except p01_error:
        pass
    else:
        raise AssertionError(f'accepted invalid retry count: {p01_invalid!r}')
print('Problem 01: passed')

Problem 01: passed


**What we learned.** Truth values can be numbers for arithmetic and equality, yet they may violate a domain-specific integer contract. Choose `type(x) is int` only when excluding integer subclasses is also desired.

---
## Problem 02 — Enum versus IntEnum: the zero-value surprise

**Scenario.** Two libraries encode states with the same numeric codes, but one uses `Enum` and the other uses `IntEnum`. Their instances behave differently in `if` statements.

**Your task.** Explain why a zero-valued enum may be true, then write an explicit state decision instead of relying on truthiness.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Create two superficially similar types

An ordinary enum has identity-based enum members. An integer enum additionally inherits integer behavior.

In [5]:
from enum import Enum, IntEnum

class P02Status(Enum):
    OFF = 0
    ON = 1

class P02LegacyStatus(IntEnum):
    OFF = 0
    ON = 1

print(bool(P02Status.OFF), bool(P02LegacyStatus.OFF))
print(P02Status.OFF == 0, P02LegacyStatus.OFF == 0)

True False
False True


**Explanation.** An ordinary `Enum` member remains true even if its `.value` is zero, because the member has no numeric truth rule. An `IntEnum` member participates in integer truth testing: its zero member is false. Equality to the integer `0` also differs between these two types.

### Step 2 — Demonstrate the accidental branch

The next function says it checks whether a status is ON, but it really checks whether the object itself is truthy.

In [6]:
def p02_broken_is_on(status: P02Status) -> bool:
    return bool(status)

print('broken OFF:', p02_broken_is_on(P02Status.OFF))

broken OFF: True


### Step 3 — Compare a domain value explicitly

Enum members are singletons within their enum type. An identity comparison states the intended business rule and does not change if the enum representation changes.

In [7]:
def p02_is_on(status: P02Status) -> bool:
    if not isinstance(status, P02Status):
        raise TypeError('expected P02Status')
    return status is P02Status.ON

print(p02_is_on(P02Status.OFF), p02_is_on(P02Status.ON))

False True


### Step 4 — Verify the representation is no longer relevant

Test all enum members and a wrong type. Explicit validation avoids accepting an unrelated integer or integer enum.

In [8]:
assert p02_is_on(P02Status.OFF) is False
assert p02_is_on(P02Status.ON) is True
for p02_bad in (0, P02LegacyStatus.ON):
    try:
        p02_is_on(p02_bad)
    except TypeError:
        pass
    else:
        raise AssertionError('wrong enum type accepted')
print('Problem 02: passed')

Problem 02: passed


**What we learned.** A zero `.value` is not a universal statement about an enum member’s truth. Model state changes and decisions using explicit enum comparisons.

---
## Problem 03 — Text that says "false" is still a nonempty string

**Scenario.** A command-line program reads flags from JSON and environment-style strings. The strings `"false"` and `"0"` are truthy, while a decoded JSON `false` is false.

**Your task.** Separate decoding from validation and implement a strict, reusable parser for textual booleans.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Observe the representation boundary

The argument to `bool()` is a string in the first cases and an already-decoded Python boolean in the JSON case.

In [9]:
import json
print(bool('false'), bool('0'), bool(''))
print(json.loads('false'), type(json.loads('false')))
print(json.loads('"false"'), type(json.loads('"false"')))

True True False
False <class 'bool'>
false <class 'str'>


**Explanation.** Nonempty strings are true regardless of what the characters *say*. `json.loads('false')` returns Python `False`, while `json.loads('"false"')` returns the nonempty string `'false'`. Treating representation as semantics is a parsing bug.

### Step 2 — Define a deliberate text language

For this API we accept `true`, `false`, `1`, and `0`, allowing case differences and surrounding whitespace. Everything else must fail rather than guess.

In [10]:
def p03_parse_bool(text: str) -> bool:
    if not isinstance(text, str):
        raise TypeError('expected text')
    normalized = text.strip().casefold()
    if normalized in ('true', '1'):
        return True
    if normalized in ('false', '0'):
        return False
    raise ValueError(f'unsupported boolean spelling: {text!r}')

for p03_text in ('true', ' FALSE ', '1', '0'):
    print(repr(p03_text), '->', p03_parse_bool(p03_text))

'true' -> True
' FALSE ' -> False
'1' -> True
'0' -> False


### Step 3 — Validate JSON after decoding

Valid JSON can contain numbers, strings, `null`, arrays, or booleans. If the application needs a JSON boolean, do not coerce all these types with `bool()`.

In [11]:
def p03_read_json_boolean(payload: str) -> bool:
    parsed = json.loads(payload)
    if type(parsed) is not bool:
        raise TypeError('JSON root must be true or false')
    return parsed

print(p03_read_json_boolean('true'))
print(p03_read_json_boolean('false'))

True
False


### Step 4 — Verify ambiguity is rejected

Confirm that zero and quoted text are not accidentally accepted as JSON boolean values. Invalid textual spellings also fail.

In [12]:
assert p03_parse_bool(' FALSE ') is False
assert p03_parse_bool('TrUe') is True
assert p03_read_json_boolean('false') is False
for p03_payload in ('0', '1', '"false"', 'null', '[]'):
    try:
        p03_read_json_boolean(p03_payload)
    except TypeError:
        pass
    else:
        raise AssertionError(f'accepted JSON non-boolean: {p03_payload}')
for p03_text in ('yes', '', 'nope'):
    try:
        p03_parse_bool(p03_text)
    except ValueError:
        pass
    else:
        raise AssertionError('unsupported spelling accepted')
print('Problem 03: passed')

Problem 03: passed


**What we learned.** Parse representations into known types before reasoning about truth. `bool(value)` answers Python truthiness, not whether text is a valid representation of a boolean.

---
## Problem 04 — A successful regex match can contain an empty string

**Scenario.** A parser extracts an optional suffix. A successful match can capture `""`; no match at all produces `None`.

**Your task.** Distinguish match existence, group participation, and nonempty group content.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — A match object is not its matched text

An empty match is still a match. We deliberately use an optional group that can participate with zero characters.

In [13]:
import re
p04_pattern = re.compile(r'item:(?P<suffix>[a-z]*)')
p04_empty = p04_pattern.fullmatch('item:')
p04_filled = p04_pattern.fullmatch('item:abc')
p04_missing = p04_pattern.fullmatch('wrong')
print(bool(p04_empty), repr(p04_empty.group('suffix')))
print(bool(p04_filled), repr(p04_filled.group('suffix')))
print(p04_missing)

True ''
True 'abc'
None


**Explanation.** A `Match` instance is true even when its matched group is empty. `None` signals that `fullmatch` found no match. These cases encode different information; `if match.group(...)` would collapse a valid empty capture into the same branch as absence.

### Step 2 — Notice a second sort of absence

A group with `*` participates and captures an empty string; a truly optional group may not participate at all, in which case its group value is `None`.

In [14]:
p04_optional = re.compile(r'item:(?P<suffix>[a-z]+)?')
p04_no_group = p04_optional.fullmatch('item:')
print(repr(p04_no_group.group('suffix')))
print(repr(p04_empty.group('suffix')))

None
''


### Step 3 — Express all three outcomes explicitly

Use `match is None` to detect a failed match, `group is None` to detect nonparticipation, and an explicit empty-string comparison for a participating but empty capture.

In [15]:
def p04_classify(pattern: re.Pattern[str], text: str) -> str:
    match = pattern.fullmatch(text)
    if match is None:
        return 'no match'
    suffix = match.group('suffix')
    if suffix is None:
        return 'group absent'
    if suffix == '':
        return 'empty suffix'
    return f'suffix={suffix}'

for p04_text in ('wrong', 'item:', 'item:abc'):
    print(p04_text, '->', p04_classify(p04_pattern, p04_text))

wrong -> no match
item: -> empty suffix
item:abc -> suffix=abc


### Step 4 — Confirm each distinction

The pattern is part of the domain definition. Test the empty and optional patterns separately.

In [16]:
assert p04_classify(p04_pattern, 'wrong') == 'no match'
assert p04_classify(p04_pattern, 'item:') == 'empty suffix'
assert p04_classify(p04_pattern, 'item:xyz') == 'suffix=xyz'
assert p04_classify(p04_optional, 'item:') == 'group absent'
print('Problem 04: passed')

Problem 04: passed


**What we learned.** For regex APIs, a match object, a missing group (`None`), and a captured empty string are three different states. Test the particular state you actually need.

---
## Problem 05 — Read all lines without confusing blank lines with end-of-file

**Scenario.** A log reader must preserve blank lines. The stream API uses the empty string `""` as the EOF sentinel, but a blank line is usually `"\
"`.

**Your task.** Build a line reader that preserves blank and whitespace-only lines and stops only at EOF.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Probe the raw stream contract

`io.StringIO` lets us demonstrate a file-like object without touching the filesystem. The stream itself remains truthy; truth testing the stream does not tell us whether data remain.

In [17]:
import io
p05_stream = io.StringIO('alpha\n\n  \nomega\n')
print(bool(p05_stream))
print(repr(p05_stream.readline()))
print(repr(p05_stream.readline()))
print(repr(p05_stream.readline()))
print(repr(p05_stream.readline()))
print(repr(p05_stream.readline()))
print(bool(p05_stream))

True
'alpha\n'
'\n'
'  \n'
'omega\n'
''
True


**Explanation.** A blank line is represented by `"\
"`, which is nonempty and true. EOF is `""`, which is false. The stream object itself is still true after exhaustion; these are two completely different objects undergoing truth testing.

### Step 2 — Preserve meaning rather than stripping too early

Calling `.strip()` *before* deciding whether a line was EOF turns `"\
"` into `""`, throwing away the distinction. Test the raw value first.

In [18]:
def p05_read_lines(stream: io.StringIO) -> list[str]:
    lines: list[str] = []
    while True:
        line = stream.readline()
        if line == '':
            break
        lines.append(line.removesuffix('\n'))
    return lines

p05_lines = p05_read_lines(io.StringIO('alpha\n\n  \nomega\n'))
print([repr(line) for line in p05_lines])

["'alpha'", "''", "'  '", "'omega'"]


### Step 3 — A compact variant using the assignment expression

The `:=` operator makes the sentinel test explicit in the loop condition. This form is safe because the condition compares the *raw line* against EOF.

In [19]:
def p05_read_lines_compact(stream: io.StringIO) -> list[str]:
    lines: list[str] = []
    while (line := stream.readline()) != '':
        lines.append(line.removesuffix('\n'))
    return lines

print(p05_read_lines_compact(io.StringIO('\nlast')))

['', 'last']


### Step 4 — Verify lossless handling of important cases

Test empty input, an entirely blank line, spaces, and a final line without a newline character.

In [20]:
for p05_input, p05_expected in [
    ('', []), ('\n', ['']), ('  \n', ['  ']),
    ('a\n\nb', ['a', '', 'b']), ('last', ['last'])
]:
    assert p05_read_lines(io.StringIO(p05_input)) == p05_expected
    assert p05_read_lines_compact(io.StringIO(p05_input)) == p05_expected
print('Problem 05: passed')

Problem 05: passed


**What we learned.** Keep the original sentinel intact until after the EOF check. A false-like transformed value may represent valid data, not exhaustion.

---
## Problem 06 — A weak reference can be true even after its target disappears

**Scenario.** A registry keeps weak references to cached objects. After the object is garbage-collected, the weak-reference object still exists and has its own truth value.

**Your task.** Check the referent correctly and avoid double-dereferencing it between the check and use.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Construct an expiring reference

A weak reference does not keep its target alive. Calling the reference returns the target or `None`. We use a deliberately short scope and a standard-library collector to make the demonstration repeatable.

In [21]:
import gc
import weakref

class P06Document:
    def __init__(self, title: str) -> None:
        self.title = title

def p06_make_reference() -> weakref.ReferenceType[P06Document]:
    document = P06Document('guide')
    return weakref.ref(document)

p06_reference = p06_make_reference()
gc.collect()
print('reference object:', bool(p06_reference))
print('live target:', p06_reference())

reference object: True
live target: None


**Explanation.** The weak-reference wrapper is a distinct object. Its `bool` result does **not** report whether the target is alive. Retrieve the target with `reference()` and check whether that returned value is `None`. The example uses a reference cycle-free object, so it can be reclaimed as soon as its last strong reference vanishes.

### Step 2 — The tempting but wrong code

A live-cache check that tests the wrapper instead of calling it always enters the branch. We model the buggy decision without trying to access a missing target.

In [22]:
def p06_wrong_is_alive(reference: weakref.ReferenceType[P06Document]) -> bool:
    return bool(reference)

print('wrong live check:', p06_wrong_is_alive(p06_reference))

wrong live check: True


### Step 3 — Fetch once, then use the strong local reference

Holding the object in a local variable keeps it alive for that operation. This is also more robust than calling `reference()` once for a check and again for use.

In [23]:
def p06_get_title(reference: weakref.ReferenceType[P06Document]) -> str | None:
    document = reference()
    if document is None:
        return None
    return document.title

p06_live = P06Document('live guide')
p06_live_reference = weakref.ref(p06_live)
print(p06_get_title(p06_live_reference))
print(p06_get_title(p06_reference))

live guide
None


### Step 4 — Verify both life-cycle states

Use an explicit deletion for the live reference to test the transition. The local object is returned or `None`; its own truth value is not the lifetime sentinel.

In [24]:
assert p06_get_title(p06_live_reference) == 'live guide'
assert p06_get_title(p06_reference) is None
del p06_live
gc.collect()
assert p06_get_title(p06_live_reference) is None
print('Problem 06: passed')

Problem 06: passed


**What we learned.** Truthiness of a wrapper does not imply validity of the resource it refers to. Dereference once and compare the result with the API’s sentinel.

---
## Problem 07 — Permission flags: `and` is not bitwise `&`

**Scenario.** A role stores permissions as an `IntFlag` bit mask. A developer uses `and` to test permission membership, accidentally returning an operand rather than computing a bit intersection.

**Your task.** Implement precise “any permission” and “all permissions” checks.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Define orthogonal permission bits

Each flag represents a separate bit. We choose `IntFlag` so that the zero flag is false and nonzero masks are true. This fact is useful but does not replace correct mask comparisons.

In [25]:
from enum import IntFlag

class P07Permission(IntFlag):
    READ = 1
    WRITE = 2
    DELETE = 4

p07_granted = P07Permission.READ | P07Permission.WRITE
print(p07_granted)
print(bool(P07Permission(0)), bool(p07_granted))

3
False True


### Step 2 — Compare logical and bitwise operators

`and` first tests its left operand and then returns one of its original operands. `&` computes a new mask containing the shared bits.

In [26]:
p07_logical = p07_granted and P07Permission.DELETE
p07_bitwise = p07_granted & P07Permission.DELETE
print('logical:', p07_logical, bool(p07_logical))
print('bitwise:', p07_bitwise, bool(p07_bitwise))

logical: 4 True
bitwise: 0 False


**Explanation.** Both operands in `granted and DELETE` are truthy, so the expression returns `DELETE` even though the granted mask does not contain that permission. The bitwise intersection is zero and correctly represents no overlap.

### Step 3 — Distinguish “any” from “all”

For “any requested permission,” test whether the intersection is nonzero. For “all requested permissions,” the intersection must equal the complete requested mask. Here an empty requested mask is intentionally vacuously satisfied for “all” and does not satisfy “any”.

In [27]:
def p07_has_any(granted: P07Permission, requested: P07Permission) -> bool:
    return bool(granted & requested)

def p07_has_all(granted: P07Permission, requested: P07Permission) -> bool:
    return (granted & requested) == requested

print('any READ or DELETE:', p07_has_any(p07_granted, P07Permission.READ | P07Permission.DELETE))
print('all READ and DELETE:', p07_has_all(p07_granted, P07Permission.READ | P07Permission.DELETE))

any READ or DELETE: True
all READ and DELETE: False


### Step 4 — Verify absent, partially present, and empty masks

Check every relevant semantic case rather than just a positive example.

In [28]:
p07_none = P07Permission(0)
assert p07_has_any(p07_granted, P07Permission.READ)
assert not p07_has_any(p07_granted, P07Permission.DELETE)
assert p07_has_any(p07_granted, P07Permission.READ | P07Permission.DELETE)
assert not p07_has_all(p07_granted, P07Permission.READ | P07Permission.DELETE)
assert p07_has_all(p07_granted, P07Permission.READ | P07Permission.WRITE)
assert p07_has_all(p07_granted, p07_none)
assert not p07_has_any(p07_granted, p07_none)
print('Problem 07: passed')

Problem 07: passed


**What we learned.** Use `&` to compute mask intersections, and say whether membership means any bit or all bits. Logical `and` is operand selection, not bitwise set intersection.

---
## Problem 08 — Chained comparisons and hidden truth tests

**Scenario.** A measurement library makes comparison return a custom “comparison report” instead of an ordinary Boolean. Python can inspect the report’s truth value when it evaluates a chain such as `low < current < high`.

**Your task.** Observe the order of operations and implement an ordinary explicit range predicate.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Return a comparison report from `<`

Rich comparison methods may return an object instead of `True`/`False`. Python only converts an intermediate result to truth when the enclosing context requires it.

In [29]:
class P08Comparison:
    def __init__(self, outcome: bool, label: str, trace: list[str]) -> None:
        self.outcome = outcome
        self.label = label
        self.trace = trace

    def __bool__(self) -> bool:
        self.trace.append(f'truth {self.label}')
        return self.outcome

class P08Number:
    def __init__(self, value: int, name: str, trace: list[str]) -> None:
        self.value, self.name, self.trace = value, name, trace

    def __lt__(self, other: 'P08Number') -> P08Comparison:
        label = f'{self.name}<{other.name}'
        self.trace.append(f'compare {label}')
        return P08Comparison(self.value < other.value, label, self.trace)

### Step 2 — Test a successful comparison chain

Watch when Python calls the first report’s `__bool__`. The last comparison result can be returned directly by the expression; `print` of the object itself does not force it to a Boolean.

In [30]:
p08_trace: list[str] = []
p08_low = P08Number(0, 'low', p08_trace)
p08_current = P08Number(5, 'current', p08_trace)
p08_high = P08Number(10, 'high', p08_trace)
p08_report = p08_low < p08_current < p08_high
print('trace immediately:', p08_trace)
print('final result type:', type(p08_report).__name__)
print('final truth:', bool(p08_report))
print('trace after bool:', p08_trace)

trace immediately: ['compare low<current', 'truth low<current', 'compare current<high']
final result type: P08Comparison
final truth: True
trace after bool: ['compare low<current', 'truth low<current', 'compare current<high', 'truth current<high']


**Explanation.** In a comparison chain, Python truth-tests the first comparison to decide whether it should continue. If that comparison succeeds, the final comparison value is returned as the expression result; this final object gets truth-tested only if its caller needs a Boolean. This is a subtler case than comparing plain integers, where every comparison already returns `bool`.

### Step 3 — Short-circuit a failed first comparison

Change the middle value so that the first comparison is false. The second comparison must never happen.

In [31]:
p08_failed_trace: list[str] = []
p08_a = P08Number(9, 'low', p08_failed_trace)
p08_b = P08Number(5, 'current', p08_failed_trace)
p08_c = P08Number(10, 'high', p08_failed_trace)
p08_failed = p08_a < p08_b < p08_c
print(bool(p08_failed))
print(p08_failed_trace)

False
['compare low<current', 'truth low<current', 'truth low<current']


### Step 4 — Write a stable public Boolean API

A public “is in range” API should return `bool` explicitly. Also decide whether the range includes its boundaries.

In [32]:
def p08_in_open_range(value: int, low: int, high: int) -> bool:
    if low >= high:
        raise ValueError('low must be less than high')
    return low < value < high

assert p08_in_open_range(5, 0, 10) is True
assert p08_in_open_range(0, 0, 10) is False
assert p08_in_open_range(10, 0, 10) is False
assert p08_in_open_range(-1, 0, 10) is False
assert p08_failed_trace == ['compare low<current', 'truth low<current', 'truth low<current']
print('Problem 08: passed')

Problem 08: passed


**What we learned.** A chained comparison uses intermediate truth testing and short-circuiting. With custom comparison return types, the whole expression may not itself be a plain `bool`; make your API contract explicit.

---
## Problem 09 — Return NotImplemented so the other operand gets a chance

**Scenario.** A custom object is compared with another custom type. An overly eager `False` result in `__eq__` prevents the other object from handling comparison.

**Your task.** Understand the difference between returning `False` and returning the special singleton `NotImplemented`.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Design a friendly operand that records attempts

The right-hand operand recognizes comparison with a left-hand “token”. The token itself does not understand other types.

In [33]:
class P09Right:
    def __init__(self, trace: list[str]) -> None:
        self.trace = trace

    def __eq__(self, other: object) -> bool:
        self.trace.append('right.__eq__')
        return isinstance(other, P09Left)

class P09Left:
    def __init__(self, trace: list[str]) -> None:
        self.trace = trace

    def __eq__(self, other: object) -> bool:
        self.trace.append('left.__eq__')
        return NotImplemented

p09_trace: list[str] = []
print(P09Left(p09_trace) == P09Right(p09_trace))
print(p09_trace)

True
['left.__eq__', 'right.__eq__']


**Explanation.** `NotImplemented` is a special singleton, **not** the same thing as `False`, `None`, or raising `NotImplementedError`. Returning it means that this operation with these operands is not supported here; Python can try the other operand’s reflected comparison. The expression eventually produces an ordinary Boolean for equality.

### Step 2 — Examine an incorrect implementation

Now the left side returns `False` immediately. Python has already received an answer and therefore does not ask the right side.

In [34]:
class P09BadLeft:
    def __init__(self, trace: list[str]) -> None:
        self.trace = trace

    def __eq__(self, other: object) -> bool:
        self.trace.append('bad_left.__eq__')
        return False

p09_bad_trace: list[str] = []
print(P09BadLeft(p09_bad_trace) == P09Right(p09_bad_trace))
print(p09_bad_trace)

False
['bad_left.__eq__']


### Step 3 — Implement a type-aware comparison

This example compares labeled tokens to other labeled tokens, but declines unsupported operand types. Use the explicit return annotation `bool | NotImplementedType` when desired; the code below avoids annotation complexity and demonstrates the runtime contract.

In [35]:
class P09Token:
    def __init__(self, label: str) -> None:
        self.label = label

    def __eq__(self, other: object):
        if not isinstance(other, P09Token):
            return NotImplemented
        return self.label == other.label

print(P09Token('a') == P09Token('a'))
print(P09Token('a') == P09Token('b'))
print(P09Token('a') == 'a')

True
False
False


### Step 4 — Test unsupported operands and symmetry

When neither side handles equality, Python’s final fallback gives an ordinary true/false outcome based on identity. Here distinct objects and strings are unequal.

In [36]:
assert (P09Token('x') == P09Token('x')) is True
assert (P09Token('x') == P09Token('y')) is False
assert (P09Token('x') == 'x') is False
assert ('x' == P09Token('x')) is False
assert p09_trace == ['left.__eq__', 'right.__eq__']
assert p09_bad_trace == ['bad_left.__eq__']
print('Problem 09: passed')

Problem 09: passed


**What we learned.** `NotImplemented` in a rich comparison is a protocol signal: it permits the other operand to respond. Returning `False` is a completed comparison, not an invitation to try again.

---
## Problem 10 — Structural matching does not treat `True` and `1` identically

**Scenario.** A validator matches a parsed value against boolean and integer cases. Although `True == 1`, the structural-pattern-matching rules treat singleton patterns such as `True` specially.

**Your task.** Write a classifier that recognizes booleans before integers, without assuming equality and pattern matching are identical.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Compare ordinary equality to pattern matching

`True == 1` is true. A `case True` pattern, however, matches the `True` singleton by identity, not by integer equality.

In [37]:
def p10_literal_match(value: object) -> str:
    match value:
        case True:
            return 'true singleton'
        case False:
            return 'false singleton'
        case 1:
            return 'integer one'
        case 0:
            return 'integer zero'
        case _:
            return 'other'

for p10_value in (True, 1, False, 0):
    print(repr(p10_value), p10_literal_match(p10_value))

True true singleton
1 integer one
False false singleton
0 integer zero


**Explanation.** Pattern matching makes a special identity check for `True`, `False`, and `None` singleton patterns. Numeric literal patterns ordinarily use equality, which is why **case order still matters**. The function above puts the singleton cases first, making its intent explicit.

### Step 2 — A broad class pattern and its limitations

`bool` inherits from `int`, so `isinstance(True, int)` is true. When the distinction matters, put the `bool` pattern first, or use exact-type guards.

In [38]:
print(isinstance(True, int))

def p10_kind(value: object) -> str:
    match value:
        case bool():
            return 'bool'
        case int():
            return 'int'
        case str():
            return 'str'
        case _:
            return 'other'

for p10_value in (True, 1, '1', 1.0):
    print(repr(p10_value), '->', p10_kind(p10_value))

True
True -> bool
1 -> int
'1' -> str
1.0 -> other


### Step 3 — Build a robust exact-type classifier

When subclasses should not count as plain integers, match a value then use guards with `type(...) is ...`. This separates structural routing from strict validation.

In [39]:
def p10_strict_kind(value: object) -> str:
    match value:
        case _ if type(value) is bool:
            return 'exact bool'
        case _ if type(value) is int:
            return 'exact int'
        case _:
            return 'unsupported'

print([p10_strict_kind(value) for value in (True, 1, False, 0, 1.5)])

['exact bool', 'exact int', 'exact bool', 'exact int', 'unsupported']


### Step 4 — Verify no integer/boolean category collisions

Test both singleton values and numeric values to show that the contract does not depend on whether a value is truthy.

In [40]:
assert p10_literal_match(True) == 'true singleton'
assert p10_literal_match(1) == 'integer one'
assert p10_literal_match(False) == 'false singleton'
assert p10_literal_match(0) == 'integer zero'
assert p10_kind(True) == 'bool'
assert p10_strict_kind(True) == 'exact bool'
assert p10_strict_kind(0) == 'exact int'
assert p10_strict_kind(2.0) == 'unsupported'
print('Problem 10: passed')

Problem 10: passed


**What we learned.** Equality, inheritance checks, and `match` patterns answer different questions. Use the test that expresses your domain: identity for singletons, class membership for families, or exact type for strict schemas.

---
## Problem 11 — NaN is truthy even though it is not a valid measurement

**Scenario.** An instrument supplies floating-point readings. A program tries `if reading` to decide whether the value is usable.

**Your task.** Distinguish numeric nonzero truth from measurement validity and write a finite-number validator.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Probe zero, NaN, and infinity

The floating-point type uses zero as its false value; being nonzero is not the same as being finite or meaningful.

In [41]:
import math
p11_values = [0.0, -0.0, 2.5, float('nan'), float('inf'), float('-inf')]
for p11_value in p11_values:
    print(repr(p11_value), 'truth:', bool(p11_value), 'finite:', math.isfinite(p11_value))

0.0 truth: False finite: True
-0.0 truth: False finite: True
2.5 truth: True finite: True
nan truth: True finite: False
inf truth: True finite: False
-inf truth: True finite: False


**Explanation.** Both signs of zero are false. NaN and positive/negative infinity are true, but `math.isfinite` rejects them. Truth tests do not implement numerical quality control. This problem concerns checking a raw measurement; it differs from the earlier notebook’s custom `Point` example.

### Step 2 — Watch a second NaN trap

NaN does not compare equal to itself. It can therefore behave unexpectedly when code assumes ordinary reflexive equality.

In [42]:
p11_nan = float('nan')
print('NaN equals itself:', p11_nan == p11_nan)
print('NaN is finite:', math.isfinite(p11_nan))

NaN equals itself: False
NaN is finite: False


### Step 3 — Define accepted measurement types and values

Here the contract accepts built-in `int` and `float`, explicitly rejects `bool`, and refuses NaN/infinity. Returning a floating-point value gives callers a normalized representation; it also means very large integers may not be representable as floats. We reject them with a clear error rather than silently overflow.

In [43]:
def p11_measurement(value: object) -> float:
    if type(value) not in (int, float):
        raise TypeError('measurement must be a plain int or float')
    try:
        normalized = float(value)
    except OverflowError as exc:
        raise ValueError('measurement is too large for a float') from exc
    if not math.isfinite(normalized):
        raise ValueError('measurement must be finite')
    return normalized

print(p11_measurement(0), p11_measurement(-1.25))

0.0 -1.25


### Step 4 — Verify false-but-valid zero and true-but-invalid NaN

This is the essential test: neither truthy nor falsy should determine admission. Type and finiteness are the actual rules.

In [44]:
assert p11_measurement(0.0) == 0.0
assert p11_measurement(-0.0) == 0.0
assert p11_measurement(4) == 4.0
for p11_bad, p11_error in [(True, TypeError), ('3', TypeError), (float('nan'), ValueError), (float('inf'), ValueError), (10**1000, ValueError)]:
    try:
        p11_measurement(p11_bad)
    except p11_error:
        pass
    else:
        raise AssertionError(f'accepted invalid measurement {p11_bad!r}')
print('Problem 11: passed')

Problem 11: passed


**What we learned.** Truthiness of a number is an answer about zero, not an answer about finite or valid input. Reject invalid numerical values using domain-specific predicates.

---
## Problem 12 — Dictionary views are live; snapshots are not

**Scenario.** A monitoring dashboard holds `dict.keys()` in one variable and `list(dict.keys())` in another. The underlying dictionary changes afterward.

**Your task.** Explain how the two collections produce different truth values as the mapping is mutated.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Capture two kinds of representations

A dictionary view reflects the underlying mapping. A list is a snapshot of the keys at the time it was constructed.

In [45]:
p12_registry: dict[str, int] = {}
p12_live_keys = p12_registry.keys()
p12_snapshot = list(p12_registry.keys())
print('initial:', bool(p12_live_keys), bool(p12_snapshot))
p12_registry['job'] = 1
print('after insert:', bool(p12_live_keys), bool(p12_snapshot))
print('contents:', list(p12_live_keys), p12_snapshot)

initial: False False
after insert: True False
contents: ['job'] []


**Explanation.** Both objects are initially false because both contain zero keys. Inserting into the original dictionary changes the **live view’s** length and truth. The previously materialized list does not update, so its truth stays false.

### Step 2 — Test after deletion

A view also sees items disappearing. A snapshot created after insertion remains nonempty even if the live dictionary subsequently empties.

In [46]:
p12_snapshot_after_insert = list(p12_live_keys)
del p12_registry['job']
print('live after delete:', bool(p12_live_keys))
print('old snapshot after delete:', bool(p12_snapshot_after_insert))

live after delete: False
old snapshot after delete: True


### Step 3 — Write two intentionally different APIs

A function that needs the current mapping state should check the mapping or live view at the moment it runs. A function that needs a historical record should explicitly capture a snapshot.

In [47]:
def p12_currently_has_jobs(registry: dict[str, int]) -> bool:
    return bool(registry)

def p12_take_key_snapshot(registry: dict[str, int]) -> tuple[str, ...]:
    return tuple(registry.keys())

p12_registry['task'] = 2
p12_saved = p12_take_key_snapshot(p12_registry)
p12_registry.clear()
print(p12_currently_has_jobs(p12_registry), bool(p12_saved))

False True


### Step 4 — Verify live versus recorded truth

Use the same underlying registry through insertion and deletion. The important result is not that one interpretation is better; it is that their meanings are different.

In [48]:
assert bool(p12_live_keys) is False
assert bool(p12_snapshot) is False
assert bool(p12_snapshot_after_insert) is True
assert p12_currently_has_jobs(p12_registry) is False
assert p12_saved == ('task',)
print('Problem 12: passed')

Problem 12: passed


**What we learned.** Truth testing a live view answers a present-state question. Truth testing a snapshot answers a historical-state question. Preserve the distinction explicitly.

---
## Problem 13 — Repeated truth tests can produce different answers

**Scenario.** A diagnostics object reads a changing condition each time it is converted to Boolean. Code checks it, then checks it again while selecting a result.

**Your task.** Demonstrate why `__bool__` should avoid hidden state changes and why an API should capture a decision once.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Reproduce a deliberately bad boolean hook

The following object toggles every time Python asks for its truth value. This is legal in terms of return type, but extremely surprising to callers.

In [49]:
class P13Flaky:
    def __init__(self) -> None:
        self.calls = 0

    def __bool__(self) -> bool:
        self.calls += 1
        return self.calls % 2 == 1

p13_flaky = P13Flaky()
print('first check:', bool(p13_flaky))
print('second check:', bool(p13_flaky))
print('calls:', p13_flaky.calls)

first check: True
second check: False
calls: 2


**Explanation.** Each `bool(obj)` invokes the hook anew. Python does not memoize the outcome. A hook that mutates state, performs IO, or relies on a changing external resource may produce contradictory results across adjacent checks. A truth hook is best kept cheap and predictable.

### Step 2 — Observe double testing in a seemingly reasonable branch

We intentionally use a fresh object so the first check is true and the second false. This demonstrates why testing an unstable object again inside the branch is dangerous.

In [50]:
p13_second = P13Flaky()
if p13_second:
    p13_message = 'ready' if p13_second else 'not ready'
else:
    p13_message = 'not ready'
print('message:', p13_message, 'calls:', p13_second.calls)

message: not ready calls: 2


### Step 3 — Improve both the object and the caller

A state holder can provide a stable Boolean property. The caller should still save a snapshot if external state may change between operations.

In [51]:
class P13ReadyState:
    def __init__(self, ready: bool) -> None:
        self.ready = ready

    def __bool__(self) -> bool:
        return self.ready

p13_state = P13ReadyState(True)
p13_decision = bool(p13_state)  # one explicit snapshot
p13_state.ready = False       # outside changes cannot rewrite the old decision
print('snapshot:', p13_decision, 'current:', bool(p13_state))

snapshot: True current: False


### Step 4 — Verify single-evaluation branching

When dealing with an object that cannot yet be fixed, taking one snapshot makes the calling code internally consistent for that decision. It does not make the underlying bad hook a good API.

In [52]:
p13_probe = P13Flaky()
p13_snapshot = bool(p13_probe)
p13_answer = 'ready' if p13_snapshot else 'not ready'
assert p13_answer == 'ready'
assert p13_probe.calls == 1
assert bool(P13ReadyState(False)) is False
assert bool(P13ReadyState(True)) is True
print('Problem 13: passed')

Problem 13: passed


**What we learned.** Boolean hooks can be evaluated repeatedly across expressions. Prefer side-effect-free `__bool__` methods and capture one value when a decision must remain internally consistent.

---
## Problem 14 — A coroutine object is not its eventual Boolean result

**Scenario.** An asynchronous function returns a Boolean after some work. A caller forgets to `await` it and writes `if check():`.

**Your task.** Show the difference between coroutine-object truth and the returned Boolean, then build a correct async branch.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Define the asynchronous check

Calling an `async def` function creates a coroutine object; its body runs when it is awaited or otherwise scheduled. It does not immediately return the declared Boolean result.

In [53]:
import asyncio

async def p14_is_allowed(user_id: int) -> bool:
    await asyncio.sleep(0)
    return user_id == 42

p14_pending = p14_is_allowed(0)
print('object type:', type(p14_pending).__name__)
print('coroutine object truth:', bool(p14_pending))
p14_pending.close()  # clean up: never leave an unawaited coroutine behind

object type: coroutine
coroutine object truth: True


**Explanation.** The coroutine object is truthy by default, regardless of the eventual value returned by the async function. Calling `if p14_is_allowed(0):` would test that coroutine object, enter the true branch, and often emit an unawaited-coroutine warning. We closed our demonstration object explicitly because it was not scheduled or awaited.

### Step 2 — Await to obtain the real Boolean

Use top-level `await`, supported by Jupyter, to obtain the coroutine result. Unlike `asyncio.run`, this form also works with the notebook kernel’s existing event loop.

In [54]:
async def p14_check_examples() -> tuple[bool, bool]:
    rejected = await p14_is_allowed(0)
    allowed = await p14_is_allowed(42)
    return rejected, allowed

p14_results = await p14_check_examples()
print(p14_results)

(False, True)


### Step 3 — Write the proper conditional

Put `await` *inside* the condition, so the `if` sees the eventual Boolean, not the coroutine wrapper.

In [55]:
async def p14_authorize(user_id: int) -> str:
    if await p14_is_allowed(user_id):
        return 'access granted'
    return 'access denied'

print(await p14_authorize(0))
print(await p14_authorize(42))

access denied
access granted


### Step 4 — Verify both branches

A negative case is essential; otherwise testing only the allowed user would accidentally conceal the missing-`await` bug.

In [56]:
assert p14_results == (False, True)
assert await p14_authorize(0) == 'access denied'
assert await p14_authorize(42) == 'access granted'
print('Problem 14: passed')

Problem 14: passed


**What we learned.** An async call and its eventual return value are distinct objects. Use `await` to get the Boolean your conditional intends to inspect; Jupyter supports top-level `await`.

---
## Problem 15 — A recursive `__bool__` implementation can never answer

**Scenario.** A custom health check tries to implement its own truth testing by calling `bool(self)`. That just calls the same hook again.

**Your task.** Reproduce the recursion safely, then build a correct delegated design with an explicit meaning for truth.

We will solve the problem in small steps. Before running each experiment, try predicting the result.

### Step 1 — Understand why the naive approach recurses

Here the hook calls `bool(self)`, which asks Python to invoke the exact same hook again. We will catch only the expected recursion error.

In [57]:
class P15Recursive:
    def __bool__(self) -> bool:
        return bool(self)

try:
    bool(P15Recursive())
except RecursionError as exc:
    print('expected:', type(exc).__name__)

expected: RecursionError


**Explanation.** The boolean protocol is dispatched whenever `bool(self)` appears, including *inside* the protocol method itself. This does not obtain any stored state; it simply recurses until Python reaches its recursion limit.

### Step 2 — Another subtle recursion through a property

The hook may look harmless when it delegates to a property, but it still recurses if that property tests the same object. We show the dependency explicitly rather than executing a second recursion example.

In [58]:
class P15IndirectBad:
    @property
    def ready(self) -> bool:
        return bool(self)  # would call __bool__ again

    def __bool__(self) -> bool:
        return self.ready

print('indirect cycle: __bool__ -> ready -> bool(self) -> __bool__')

indirect cycle: __bool__ -> ready -> bool(self) -> __bool__


### Step 3 — Define a single source of truth

Put the actual state in an ordinary data attribute. The hook should derive a real Boolean from that state without calling `bool(self)` or another method that cycles back into it.

In [59]:
class P15Health:
    def __init__(self, failures: int) -> None:
        if type(failures) is not int or failures < 0:
            raise ValueError('failures must be a nonnegative plain integer')
        self.failures = failures

    @property
    def ready(self) -> bool:
        return self.failures == 0

    def __bool__(self) -> bool:
        return self.ready

p15_healthy = P15Health(0)
p15_unhealthy = P15Health(2)
print(bool(p15_healthy), bool(p15_unhealthy))

True False


### Step 4 — Verify after a state transition

The object is deliberately mutable. The hook reads its state but does not mutate it, so successive checks agree until a real state change occurs.

In [60]:
assert bool(p15_healthy) is True
assert bool(p15_unhealthy) is False
assert p15_unhealthy.ready is False
p15_unhealthy.failures = 0
assert p15_unhealthy.ready is True
assert bool(p15_unhealthy) is True
try:
    P15Health(True)
except ValueError:
    pass
else:
    raise AssertionError('boolean failure count accepted')
print('Problem 15: passed')

Problem 15: passed


**What we learned.** Never implement `__bool__` in terms of truth testing the same object. Delegate to independent underlying state, and avoid indirect cycles through helpers or properties.

---
# Final synthesis — What connects these problems?

The introductory lesson establishes *how Python chooses the truth value of an object*. The examples here show why that mechanism must not stand in for a more specific question:

- `bool` and `int` are related types, but **type validity** is not truthiness (Problems 1, 3, 10, 11).
- An enum member, a match object, a weak reference, a stream, and a coroutine are **wrappers or control objects**. Test the state or value you actually mean (Problems 2, 4, 5, 6, 14).
- A bit mask needs bitwise operations; a chained comparison may truth-test intermediate results; `NotImplemented` delegates a comparison rather than returning false (Problems 7–9).
- A live view is not a frozen snapshot, and truth hooks can be called again. Make temporal behavior and side effects explicit (Problems 12–13).
- A custom `__bool__` must ultimately derive a real Boolean from independent state rather than recursively testing itself (Problem 15).

**A useful code-review question:** “What does this particular truth test assert—nonzero, nonempty, present, valid, alive, authorized, completed, or something else?” If more than one interpretation is plausible, use a named predicate or explicit comparison instead.

## Optional self-check: consolidate the main pitfalls

Before viewing the next cell, predict all six results. Notice that these examples are about six different *meanings* of truth, even though they all use `bool(...)` or comparisons.

1. `bool('false')`
2. `bool(P02Status.OFF)`
3. `bool(P07Permission.READ & P07Permission.DELETE)`
4. `bool(float('nan'))`
5. `p04_pattern.fullmatch('item:') is None`
6. `p01_retry_count(True)` (result or exception?)

In [61]:
assert bool('false') is True
assert bool(P02Status.OFF) is True
assert bool(P07Permission.READ & P07Permission.DELETE) is False
assert bool(float('nan')) is True
assert (p04_pattern.fullmatch('item:') is None) is False
try:
    p01_retry_count(True)
except TypeError:
    print('strict integer validator rejected True as intended')
else:
    raise AssertionError('expected TypeError')
print('Final self-check: passed')

strict integer validator rejected True as intended
Final self-check: passed
